# Student Performance Statistics and Linear Regression Analysis
## Academic Performance Evaluation using Pandas, Seaborn, and Scikit-Learn

**Objective:**
* Perform comprehensive descriptive and inferential statistics on student datasets.
* Interpret correlation coefficients to analyze social and environmental influences on grades.
* Build and evaluate a linear regression model to predict final academic scores.

# Step 1: Library Setup and Dataset Loading
In this step, we import the required analytical modules and read the dataset using custom separation parameters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load the dataset with semicolon separator as specified in instructions
df = pd.read_csv('student-por.csv', sep=';')
print("Dataset successfully loaded into DataFrame 'df'.")

# Step 2: Dataset Exploration
Let's extract dimensional metrics, attribute structures, and check clean data rules.

In [ ]:
# Task 1: How many rows and columns are present?
rows, cols = df.shape
print(f"Total Rows: {rows}")
print(f"Total Columns: {cols}")

# Task 2: How many numerical and categorical columns are there?
num_cols = df.select_dtypes(include=[np.number]).shape[1]
cat_cols = df.select_dtypes(exclude=[np.number]).shape[1]
print(f"Numerical Columns: {num_cols}")
print(f"Categorical Columns: {cat_cols}")

# Task 3: Check for missing values
missing_count = df.isnull().sum().sum()
print(f"Total Missing Values: {missing_count}")

# Task 4: Check for duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Total Duplicate Rows: {duplicate_count}")

### Dataset Exploration Summary Analysis:
1. **Dimensions:** The dataset contains **649 rows** and **33 columns**.
2. **Column Layouts:** There are **16 numerical attributes** and **17 categorical attributes**.
3. **Missing Values:** There are **0 missing fields**, indicating a perfectly complete collection.
4. **Duplicates:** There are **0 duplicate rows** present within the parsed source.

# Step 3: Descriptive Statistics
## Section A: Measures of Central Tendency

In [ ]:
# Overall Central Tendency metrics for Final Grade (G3)
g3_mean = df['G3'].mean()
g3_median = df['G3'].median()
g3_mode = df['G3'].mode()[0]

print(f"Overall G3 Mean: {g3_mean:.2f}")
print(f"Overall G3 Median: {g3_median:.2f}")
print(f"Overall G3 Mode: {g3_mode}")

print("\n--- School-Specific Comparison Breakdown ---")
school_comparison = df.groupby('school')['G3'].agg(['mean', 'median'])
print(school_comparison)

### Central Tendency Analytical Interpretation:
* **Overall Distribution:** The student community exhibits a mean final score of **11.91**, a median of **12.00**, and a most frequent grade (mode) of **11**.
* **School Differences:** Students attending Gabriel Pereira (`GP`) yield a slightly higher average performance (**Mean: 11.98, Median: 12.0**) compared to students at Mousinho da Silveira (`MS`), who show a **Mean: 11.77, Median: 11.0**. This highlights a mild structural divergence in performance distribution across the campuses.

## Section B: Measures of Variability

In [ ]:
g3_std = df['G3'].std()
g3_var = df['G3'].var()
g3_range = df['G3'].max() - df['G3'].min()
q3, q1 = np.percentile(df['G3'], [75 ,25])
g3_iqr = q3 - q1

print(f"Standard Deviation: {g3_std:.2f}")
print(f"Variance: {g3_var:.2f}")
print(f"Range: {g3_range}")
print(f"Interquartile Range (IQR): {g3_iqr:.2f}")

### Variability Interpretation:
* The **Standard Deviation (3.23)** and **Variance (10.43)** show that student performance varies moderately around the mean grade.
* The absolute **Range is 20**, meaning scores cover the entire span from minimum possible to maximum possible marks. 
* An **IQR of 4.00** shows that the middle 50% of students fall within a compact band of 4 points, showcasing consistency among typical performers.

## Section C: Graphical Representation

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Histogram Distribution of G3
sns.histplot(df['G3'], kde=True, ax=axes[0], color='royalblue', bins=15)
axes[0].set_title('Distribution Pattern of Final Grades (G3)', fontsize=14)
axes[0].set_xlabel('Final Grade (G3)')
axes[0].set_ylabel('Student Density Count')

# Plot 2: Boxplot comparing G3 across study time groups
sns.boxplot(x='studytime', y='G3', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Final Grade (G3) Variation Across Study Time Groups', fontsize=14)
axes[1].set_xlabel('Weekly Study Time Group (1: <2h, 2: 2-5h, 3: 5-10h, 4: >10h)')
axes[1].set_ylabel('Final Grade (G3)')

plt.tight_layout()
plt.show()

### Graph Observations and Interpretations:
1. **G3 Grade Distribution:** The histogram shows a mostly normal distribution concentrated between 10 and 14, with a small secondary peak at 0. This peak likely represents students who missed exams or dropped the course.
2. **Study Time Boxplot:** There is a clear upward trend in grades as study time increases. Students in higher weekly study brackets (groups 3 and 4) maintain higher median scores and higher lower-quartile ranges compared to group 1.

# Step 4: Inferential Statistics
## Section A: Pearson Correlation Calculations

In [ ]:
# Calculate correlations against the dependent metric
target_vars = ['freetime', 'absences', 'studytime', 'traveltime']
correlations = df[target_vars + ['G3']].corr()['G3'].drop('G3')

print("--- Pearson Correlation Coefficient Values against G3 ---")
print(correlations)

### Correlation Factor Explanations:
* **`studytime` (+0.2497):** Shows a clear, weak positive correlation. Spending more time studying is the strongest positive driver for higher final grades among these features.
* **`freetime` (-0.1227):** Shows a weak negative correlation. This indicates that excessive unmonitored free time may take away from academic focus.
* **`traveltime` (-0.1026):** Shows a weak negative correlation. Longer commute times likely contribute to student fatigue and less available study time.
* **`absences` (-0.0914):** Shows a very slight negative correlation. Missing classes has a negative impact on grade retention, though it is minor in this specific dataset.

# Step 5: Linear Regression Modeling

In [ ]:
# Isolate independent variables (X) and dependent target variable (y)
X = df[['freetime', 'absences', 'studytime', 'traveltime']]
y = df['G3']

# Split into 70% Training and 30% Testing sets using a fixed seed for consistency
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# Initialize and fit model
model = LinearRegression()
model.fit(X_train, y_train)

# Execute test predictions
y_pred = model.predict(X_test)

# Compute verification statistics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Evaluation Statistics ---")
print(f"Intercept: {model.intercept_:.4f}")
for feature, coef in zip(X.columns, model.coef_):
    print(f"Coefficient for {feature}: {coef:.4f}")

print(f"\nMean Squared Error (MSE) on Test Set: {mse:.4f}")
print(f"R-squared (R2) Accuracy Score on Test Set: {r2:.4f}")

### Linear Regression Insights and Interpretations:
1. **Coefficient Interpretations:**
    * For each unit increase in **study time group**, final grades are expected to increase by **0.7410** points, keeping all other variables constant.
    * **Absences (-0.0315)**, **free time (-0.2312)**, and **travel time (-0.3670)** all have negative impacts on the final grade.
2. **Mean Squared Error (MSE - 8.8752):** On average, the squared difference between the model's predictions and actual grades is roughly 8.88 points.
3. **R-squared (0.0634):** The model has an R-squared value of **6.34%**. This means that lifestyle factors like study time, travel time, absences, and free time explain about 6.34% of the variance in a student's final grade. Other factors like prior grades (`G1`, `G2`), family support, and school resources likely account for the remaining variance.